# Job Listing Prioritization for Career Changers: EDA/SDA 3 (Text features)

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

**What do the listing descriptions say?** This notebook turns the description text of EDA_SDA_2's scope listings into **features**, and validates the text rules that hide listings, so `EDA_SDA_4` can test every feature (text included) and the app can hide what a career changer can't get. **It builds no matching models:** résumé-to-listing fit (keyword overlap, TF-IDF, BGE embeddings, RRF) is in `ML_Fit`, which uses this notebook's cleaned text and skills. A text feature is kept only if it adds information beyond the non-text features.

**For career changers,** the most important text features are what a listing asks of a newcomer (experience, education, licenses, training) and which of its skills carry over from other fields.

**Inputs:** EDA_SDA_2's scope table, plus the description text for those listings, loaded from the saved flattened parts (EDA_SDA_1's table keeps only description features, to fit in memory).

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the data, and what's the scope? | Final table 1 |
| EDA_SDA_2 | Which listings do we study? The scope cut, additional cleaning, non-text checks, ghost listings | Scope table, non-text features, occupation summary |
| EDA_SDA_3 | What do the descriptions say? Text features, validated blockers, skills mapped to O*NET (no matching models) | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

**View:** listings with a description (77% of the scope; EDA_SDA_2 3.9). Text features are **unknown** for the rest, never "no", and "has a description" stays a standard confounder: listings without one close in about 3 days.

**Data rules:** aggregates only; never print descriptions, titles or sample rows, except in cells tagged `data-rows` (review cells for a teammate to check locally; cleared before every commit). Text stays local: no listing text goes to outside AI services. Clear outputs before committing.

### Clarifying questions
- What does description length actually capture: employer intent, or boilerplate?
- Why do listings with urgency wording stay open longer?
- Can evergreen wording ("always accepting applications") identify ghost listings?
- Are EDA_SDA_1's career-change flags (years required, degree required, training and career-changer wording) accurate, and what do they miss? Two are urgent: "asks 5+ years" would hide 31% of open listings, and the degree flag fires on only 3–6% (EDA_SDA_2 3.11).
- Which requirements are blockers for everyone (hide the listing), and which depend on the applicant (degree, certification)?
- Which skills does each listing ask for, mapped to O*NET skills, so `ML_Fit` can match on skills rather than words?

### Roadmap

1. **Setup:** the scope table and its description text
2. **Description cleanup:** HTML to structured text, boilerplate removal, normalization and lemmas
3. **The blocker rules:** years of experience (required vs. preferred) and degree requirements
4. **Other text features:** evergreen wording, hiring-difficulty phrases, licenses and certifications, training and career-changer wording, urgency, remote, length
5. **Validation:** precision and recall of each rule on hand-checked samples
6. **Skills mapped to O*NET** (for `ML_Fit`)
7. **Final table:** validated blockers applied (hidden, never removed), saved for EDA_SDA_4 and `ML_Fit`
8. **Conclusion**


## Plan: text features to build

| Feature | Question | Notes |
|:--|:--|:--|
| **Description cleanup** (moved here from EDA_SDA_2 2b) | Strip HTML and boilerplate (equal-opportunity statements, benefits, "about us") | `desc_clean` and `desc_words_clean`; the base for every feature below and for `ML_Fit` |
| Description length | Is it employer intent, or just boilerplate? | Measured on `desc_clean` |
| Urgency wording | Why do "urgent" listings stay open *longer*? | May mark hard-to-fill roles |
| Remote mentions | Does remote add beyond `contract_types`? | |
| **Evergreen wording** | Listings that collect applications instead of filling one job: a ghost-listing signal | Six phrase groups, each scored and checked on its own: **talent pool** ("join our talent community", "future opportunities"), **no specific opening** ("general application", "not for a specific position"), **always hiring** ("always accepting applications", "rolling basis"), **many openings in one ad** ("multiple openings", "various locations"), **pipeline labels** ("pipeline requisition", "evergreen requisition"), **staffing-style** ("on behalf of our client"); plus a generic-title check ("Software Engineer – All Levels"). Compared with EDA_SDA_2's likely-ghost flag (3.1): kept only if it adds to "old and from a quiet company" |
| Hiring-difficulty phrases | "Sign-on bonus", "hiring immediately", "relocation assistance", "will train" | Pairs with reposts and listing age |
| Years of experience required | "3+ years": is EDA_SDA_1's rule accurate? Required or preferred? In the field, or any experience? | The most common blocker for career changers |
| Education required | "Bachelor's required" vs. "or equivalent experience"; field-specific degrees | EDA_SDA_1's pattern only fires when "required" comes within six words of "bachelor's", so the most common wording ("Bachelor's degree in Computer Science or a related field") never matches: it flags only 3–6% of listings. Rebuilt in section 3, with requirements and preferred sections told apart |
| **Clearance, sponsorship, travel, shifts** (added 2026-10-10) | Security clearance required; "no visa sponsorship"; travel percentage; shift or on-call work; contract-to-hire | Clearance and sponsorship are hard blockers for some career changers: hidden per applicant in the app, like a missing degree |
| Licenses and certifications required | "Security+", "CCNA", "CISSP" and the like | A required credential the career changer lacks is a blocker |
| Training and career-changer wording | "Will train", "paid training", "career changers welcome", "transferable skills" | Validates EDA_SDA_1's flags: a team member checks a random sample of matches locally (never shared with AI tools) |
| **Hard-blocker rules** | Which years-asked threshold and which required credentials hide a listing from everyone, per role bucket? | Validated here, then applied in this notebook's final table, with a count of the listings each rule hides (hidden in the app, never removed) |
| Required vs. preferred qualifications | Split on headings like "Requirements" and "Nice to have" | Part of fit and gap analysis in `ML_Fit` |
| **Skills, mapped to O*NET** | Extracted with spaCy (PredictLeads `tags` are sparse) and mapped to O*NET skills | `ML_Fit` matches résumés to listings on these skills, so "transferable" doesn't depend on the old field's vocabulary |

**Moved to `ML_Fit`:** role similarity, TF-IDF and embeddings for matching. **Moved to `ML_Rank`:** TF-IDF terms that predict how fast a listing closes (a text input to the timing model). Topic modeling stays a stretch goal.

## Plan: preprocessing

Course standard (MLforTexts): normalize text (lowercase, regex to keep letters), remove NLTK stopwords, lemmatize with spaCy. Advanced: remove boilerplate where headings make it detectable. Any vocabulary learned from the text is fit on the training months only.

**Evaluation (course standard):** each validated rule reports precision and recall on a hand-checked random sample (checked locally by a teammate), against the keyword rule from EDA_SDA_1 as the baseline.


## 1. Setup

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts)
import gc
import html
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, INK, INK_2, MUTED, CLEAN_DIR, PART2_TABLE,
    bar_h, finish, heatmap, km_median, show, use_chart_style,
)
from src.pl_data import JOBS_N, map_parts  # saved flattened parts, loaded one at a time
use_chart_style()
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)  # short plain-text descriptions look like file names to bs4

TRAIN_END = pd.Period("2026-07", "M")  # anything learned from the text is fit on April–July only (ML_Rank's training months)
TEXT_CACHE = CLEAN_DIR / "eda3_text.pkl"  # cleaned text, so reruns skip the slow steps; git-ignored like all of data/

In [ ]:
# 1b Load EDA_SDA_2's scope table and the description text for its listings (from the saved flattened parts,
# one part at a time). Only counts are shown.
jobs = pd.read_pickle(PART2_TABLE)
ids = set(jobs.id)
raw = map_parts(f"job_openings_{JOBS_N}", lambda d: d.loc[d["id"].isin(ids), ["id", "attributes.description"]])
raw = raw.drop_duplicates("id").set_index("id")["attributes.description"]
jobs["desc_raw"] = jobs.id.map(raw)
del raw
gc.collect()
show("1b The scope table and its text", pd.DataFrame({"value": {
    "listings (EDA_SDA_2 scope table)": f"{len(jobs):,}",
    "columns": jobs.shape[1] - 1,
    "with description text in the parts": f"{jobs.desc_raw.fillna('').str.strip().ne('').sum():,}",
    "has_desc (EDA_SDA_1) agrees": f"{(jobs.desc_raw.fillna('').str.strip().ne('') == jobs.has_desc).mean():.1%}",
    "first seen in the training months (April–July)": f"{(jobs.month <= TRAIN_END).sum():,}",
}}))

## 2. Description cleanup

**Why:** every text feature, and `ML_Fit`'s matching, should read what the listing asks for, not the legal and benefits text most listings share. Three steps, each with its result reported:

| Step | What | Output |
|:--|:--|:--|
| 2a | HTML to text, one line per paragraph, list item or heading (long unbroken text split into sentences) | Lines per listing |
| 2b | Remove boilerplate line by line: known patterns (equal opportunity and legal, benefits, application logistics), "About the company" sections, and text shared word for word across 5+ employers (learned on April–July only; lines with a requirement cue are never cut) | `desc_clean`, `words_clean` |
| 2c | Course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas | `desc_lemmas` (for word counts and TF-IDF later) |

The rules in sections 3–4 run on `desc_clean`, which keeps case, numbers and punctuation ("3+ years", "B.S.", "Security+"). The cleaned text is cached in `data/clean/` so reruns skip the slow steps.

In [ ]:
# 2a HTML to text, keeping the structure: each paragraph, list item or line break becomes its own line, so headings
# ("Requirements", "About us") and bullet points survive for 2b and for splitting required from preferred (section 3).
# EDA_SDA_1's clean_text flattened everything to one line, which hides that structure.
BLOCK_TAGS = ["p", "li", "br", "div", "h1", "h2", "h3", "h4", "h5", "h6", "tr", "ul", "ol"]

def to_lines(s):
    """Description HTML -> a list of clean text lines (units). Long unbroken text is split into sentences."""
    if not isinstance(s, str) or not s.strip():
        return []
    if "<" in s and ">" in s:
        soup = BeautifulSoup(s, "html.parser")
        for tag in soup.find_all(BLOCK_TAGS):
            tag.insert_before("\n")
            tag.insert_after("\n")
        s = soup.get_text()
    s = html.unescape(s).replace("\r", "\n")
    units = []
    for line in s.split("\n"):
        line = " ".join(line.replace("•", " ").split()).strip(" -*·")
        if not line:
            continue
        if len(line.split()) > 120:  # one long block: split into sentences so boilerplate can be cut without the rest
            units += [x.strip() for x in re.split(r"(?<=[.!?])\s+(?=[A-Z])", line) if x.strip()]
        else:
            units.append(line)
    return units

if TEXT_CACHE.exists() and set(pd.read_pickle(TEXT_CACHE).index) == ids:
    text = pd.read_pickle(TEXT_CACHE)
    print(f"loaded the cleaned text from {TEXT_CACHE.name} (delete it to rebuild 2a–2c)")
else:
    text = pd.DataFrame({"units": jobs.set_index("id").desc_raw.map(to_lines)})
    text["month"] = jobs.set_index("id").month
    text["company_id"] = jobs.set_index("id").company_id
    text["words_raw"] = text.units.map(lambda u: sum(len(x.split()) for x in u))
jobs = jobs.drop(columns="desc_raw", errors="ignore")
show("2a Description text after removing HTML (lines per listing and words)", pd.DataFrame({
    "lines": text.units.map(len).describe(percentiles=[0.1, 0.5, 0.9]),
    "words": text.words_raw.describe(percentiles=[0.1, 0.5, 0.9]),
}).drop(index="count").round(1))

In [ ]:
# 2b Boilerplate removal, line by line, three ways:
#   1. Known boilerplate: equal-opportunity and legal statements, benefits, application logistics (patterns below).
#   2. "About the company" sections: from an "About us"-style heading to the next heading ("About the role" is kept).
#   3. Shared template text: a line of 8+ words that appears word for word in listings from 5+ companies, learned
#      from the training months only (April–July) and then removed everywhere. One company repeating its own
#      requirements isn't cut (only text shared across employers), and neither is any line with a requirement cue:
#      generic requirements ("Bachelor's degree in Computer Science or a related field") are shared across employers
#      too, and sections 3–4 need them.
BOILERPLATE = {
    "equal opportunity and legal": r"equal (?:employment )?opportunit|without regard to|protected (?:veteran|class)|affirmative action"
                                   r"|e-?verify|reasonable accommodation|\beeoc?\b|nondiscrimination|fair chance|background check"
                                   r"|drug[- ]free|at[- ]will|privacy (?:notice|policy)|recruitment (?:fraud|scam)|unsolicited",
    "benefits": r"\b401\s*\(?k\)?|\bdental\b|vision (?:insurance|coverage)|paid time off|\bpto\b|parental leave|tuition (?:reimbursement|assistance)"
                r"|employee assistance|wellness (?:program|stipend)|benefits? (?:include|package|offered)|health (?:insurance|benefits)"
                r"|life insurance|retirement (?:plan|savings)|paid holidays",
    "application logistics": r"click (?:apply|here)|apply (?:now|today|online)|submit (?:your )?(?:resume|application)"
                             r"|to apply,|applications? (?:will be )?accepted until",
}
HEADING = re.compile(r"^[A-Z][\w &/',’-]{1,60}:?$")  # a short line, starting with a capital letter
ABOUT_HEAD = re.compile(r"^(?:about (?!the (?:role|job|position|team|opportunity))[\w&.,' -]{1,40}|who we are|our (?:company|mission|story|culture|values)"
                        r"|company (?:overview|description))\s*:?$", re.IGNORECASE)
MIN_TEMPLATE_WORDS, MIN_TEMPLATE_COMPANIES = 8, 5
REQUIREMENT_CUE = re.compile(r"\b(?:years?|yrs|experience|degree|bachelor|master|associate|diploma|certif\w*|licen[sc]\w*|clearance"
                             r"|required|requirements?|preferred|qualifications?|proficien\w*|knowledge|ability|skills?|familiar\w*"
                             r"|responsib\w*|you will|must)\b|\d\s*\+", re.IGNORECASE)

def norm_line(x):
    return " ".join(re.sub(r"[^a-z ]", " ", x.lower()).split())

if "reasons" not in text:
    # 3. Shared template text, learned on the training months
    train = text[text.month <= TRAIN_END]
    pairs = pd.DataFrame([(norm_line(x), c) for u, c in zip(train.units, train.company_id) for x in set(u)
                          if len(x.split()) >= MIN_TEMPLATE_WORDS], columns=["line", "company_id"])
    TEMPLATE_LINES = set(pairs.groupby("line").company_id.nunique().loc[lambda s: s >= MIN_TEMPLATE_COMPANIES].index)
    del pairs, train

    def classify(units):
        """For each line: '' (kept) or the reason it's removed."""
        out, in_about = [], False
        for x in units:
            is_heading = len(x.split()) <= 6 and bool(HEADING.match(x))
            if is_heading:
                in_about = bool(ABOUT_HEAD.match(x))
            if in_about:
                out.append("about the company")
                continue
            reason = next((k for k, pat in BOILERPLATE.items() if re.search(pat, x, re.IGNORECASE)), "")
            if (not reason and len(x.split()) >= MIN_TEMPLATE_WORDS and norm_line(x) in TEMPLATE_LINES
                    and not REQUIREMENT_CUE.search(x)):
                reason = "shared template text"
            out.append(reason)
        return out

    text["reasons"] = text.units.map(classify)
    text["kept"] = [[x for x, r in zip(u, rs) if not r] for u, rs in zip(text.units, text.reasons)]
    text["desc_clean"] = text.kept.map("\n".join)
    text["words_clean"] = text.kept.map(lambda u: sum(len(x.split()) for x in u))
    print(f"shared template lines learned on April–July: {len(TEMPLATE_LINES):,}")

removed = pd.DataFrame([(r, len(x.split())) for u, rs in zip(text.units, text.reasons) for x, r in zip(u, rs) if r],
                       columns=["reason", "words"])
has_text = text.words_raw > 0
by_reason = removed.groupby("reason").words.sum()
show("2b Words removed as boilerplate (listings with a description)", pd.DataFrame({
    "words removed": by_reason,
    "share of all words": by_reason / text.words_raw.sum(),
    "listings with any": pd.Series({r: text.reasons.map(lambda rs, r=r: r in rs).sum() for r in by_reason.index}),
}).sort_values("words removed", ascending=False).round(3))
jobs = jobs.drop(columns=["words_raw", "words_clean"], errors="ignore").merge(
    text[["words_raw", "words_clean"]], left_on="id", right_index=True, how="left")
src = jobs[jobs.words_raw > 0].groupby("source_type")
show("2b Share of each description kept after cleanup, by source (median per listing)", pd.DataFrame({
    "median words before": src.words_raw.median(), "median words after": src.words_clean.median(),
    "median share kept": src.apply(lambda g: (g.words_clean / g.words_raw).median(), include_groups=False),
    "cleaned to nothing": src.apply(lambda g: g.words_clean.eq(0).mean(), include_groups=False),
    "listings": src.size(),
}).round(3))

In [ ]:
# 2b review: a few random listings before and after cleanup, for a teammate to check locally that 2b cuts boilerplate
# and keeps requirements. Shows real listing text. Tagged "data-rows": clear this output before committing.
for i in text[has_text].sample(3, random_state=7).index:
    print("=" * 100, "\nBEFORE:\n", "\n".join(text.at[i, "units"])[:2500], "\n\nAFTER:\n", text.at[i, "desc_clean"][:2500])

In [ ]:
# 2c Normalize and lemmatize (course standard: lowercase, letters only, NLTK stopwords, spaCy lemmas). desc_lemmas is
# the input for word counts and TF-IDF in ML_Fit and ML_Rank; the rules in sections 3–4 run on desc_clean, which
# keeps case, numbers and punctuation ("3+ years", "B.S.", "CompTIA Security+").
import nltk
import spacy
try:
    STOP = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)  # one-time download of NLTK's word list (no project data leaves the laptop)
    STOP = set(nltk.corpus.stopwords.words("english"))

if "desc_lemmas" not in text:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # the tagger and lemmatizer are all that's needed
    lemmas = []
    for doc in nlp.pipe(text.desc_clean.str.replace("\n", ". ").str[:20000], batch_size=64):
        lemmas.append(" ".join(t.lemma_.lower() for t in doc if t.is_alpha and t.lemma_.lower() not in STOP))
    text["desc_lemmas"] = lemmas
    text.drop(columns="kept").to_pickle(TEXT_CACHE)
    print(f"saved {TEXT_CACHE.name}")
words_lem = text.desc_lemmas.str.split().str.len()
show("2c Words per description at each step (listings with a description)", pd.DataFrame({
    "after HTML removal": text.words_raw[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "after boilerplate removal": text.words_clean[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
    "lemmas, stopwords removed": words_lem[has_text].describe(percentiles=[0.1, 0.5, 0.9]),
}).drop(index="count").round(0))

In [ ]:
# 2 Chart: words per description at each cleanup step (left), and what boilerplate removal takes out (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
bins = np.logspace(0, np.log10(max(text.words_raw.max(), 10)), 40)
for vals, label, color in [(text.words_raw[has_text], "after HTML removal", MUTED), (text.words_clean[has_text], "after boilerplate removal", BLUE),
                           (words_lem[has_text], "lemmas, stopwords removed", ORANGE)]:
    ax1.hist(vals.clip(lower=1), bins=bins, histtype="step", lw=1.8, color=color, label=label)
ax1.set(xscale="log", title="Words per description at each step", xlabel="words (log scale)", ylabel="listings")
ax1.legend(fontsize=8, frameon=False)
bar_h(ax2, 100 * by_reason / text.words_raw.sum(), "Share of all description words removed",
      "% of words", "{:.1f}%")
finish()

## 3. The blocker rules: years of experience and degree

_To write next._ Years asked, split into required vs. preferred (sections found from headings kept in 2a) and in-field vs. any experience; the 5+ years threshold rechecked per role bucket. Degree requirements rebuilt (the most common wording, "Bachelor's degree in … or a related field", isn't caught today), with "or equivalent experience" alongside.

## 4. Other text features

_To do._ Evergreen wording (six phrase groups plus a generic-title check, compared with EDA_SDA_2's likely-ghost flag), hiring-difficulty phrases, licenses and certifications, training and career-changer wording, urgency, remote and description length on the cleaned text.

## 5. Validation

_To do._ For each rule, a random sample of matches and non-matches (stratified by role bucket) is saved to `data/` for a teammate to mark right or wrong locally; the notebook reads the marks back and reports precision and recall against EDA_SDA_1's rule as the baseline.

## 6. Skills mapped to O*NET

_To do._ Needs the O*NET skills download (terms of use first). Skills extracted with spaCy, mapped to O*NET skills for `ML_Fit`.

## 7. Final table

_To do._ Validated blockers applied (hidden in the app, never removed), with a count of what each hides; saved for EDA_SDA_4 and `ML_Fit`.

## 8. Conclusion

_To do._